In [1]:
# load dataset
import pandas as pd
df = pd.read_excel("Crop_recommendation.xlsx")
df.head()


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [2]:
# Preprocess (encode + scale + split)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf

X = df.drop(columns=["label"])
y = df["label"]

le = LabelEncoder()
y_enc = le.fit_transform(y)

y_ohe = tf.keras.utils.to_categorical(y_enc)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_ohe, test_size=0.2, random_state=42, stratify=y_enc
)


In [3]:
#Evaluation Helper
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

def evaluate(model, X_test, y_test):
    y_true = np.argmax(y_test, axis=1)
    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)
    acc = accuracy_score(y_true, y_pred)
    print("Accuracy:", acc)
    print(classification_report(y_true, y_pred, target_names=le.classes_))
    return acc


In [4]:
#ALGO 1 — DNN / MLP
#Build MLP
from tensorflow.keras import layers, models

input_dim = X_train.shape[1]
num_classes = y_train.shape[1]

mlp = models.Sequential([
    layers.Dense(128, activation='relu', input_shape=(input_dim,)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

mlp.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
mlp.summary()


c:\Users\ashee\Desktop\crop project\venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 22)             │         1,430 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,710 (41.84 KB)

 Trainable params: 10,710 (41.84 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
#Train MLP
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

mlp.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=80,
    batch_size=32,
    callbacks=[es],
    verbose=2
)

acc_mlp = evaluate(mlp, X_test, y_test)


Epoch 1/80
50/50 - 1s - 15ms/step - accuracy: 0.1730 - loss: 2.7938 - val_accuracy: 0.5057 - val_loss: 2.2301
Epoch 2/80
50/50 - 0s - 2ms/step - accuracy: 0.4634 - loss: 1.9921 - val_accuracy: 0.6705 - val_loss: 1.3418
Epoch 3/80
50/50 - 0s - 2ms/step - accuracy: 0.6155 - loss: 1.3143 - val_accuracy: 0.8807 - val_loss: 0.7685
Epoch 4/80
50/50 - 0s - 2ms/step - accuracy: 0.7052 - loss: 0.9221 - val_accuracy: 0.8068 - val_loss: 0.5401
Epoch 5/80
50/50 - 0s - 2ms/step - accuracy: 0.7588 - loss: 0.7178 - val_accuracy: 0.8920 - val_loss: 0.4082
Epoch 6/80
50/50 - 0s - 2ms/step - accuracy: 0.7860 - loss: 0.6088 - val_accuracy: 0.9318 - val_loss: 0.3283
Epoch 7/80
50/50 - 0s - 2ms/step - accuracy: 0.8226 - loss: 0.5342 - val_accuracy: 0.9318 - val_loss: 0.2860
Epoch 8/80
50/50 - 0s - 2ms/step - accuracy: 0.8245 - loss: 0.4826 - val_accuracy: 0.9489 - val_loss: 0.2442
Epoch 9/80
50/50 - 0s - 2ms/step - accuracy: 0.8523 - loss: 0.4319 - val_accuracy: 0.9261 - val_loss: 0.2302
Epoch 10/80
50/50 

In [6]:
#tabnet algo
#Prepare Data For TabNet
from pytorch_tabnet.tab_model import TabNetClassifier
import numpy as np

X_raw = df.drop(columns=["label"]).values
y_raw = y_enc

from sklearn.model_selection import train_test_split
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)


In [15]:
#Train TabNet
tabnet = TabNetClassifier()

tabnet.fit(
    X_train_t, y_train_t,
    eval_set=[(X_test_t, y_test_t)],
    patience=10,
    max_epochs=100
)
acc_tabnet = accuracy_score(y_test_t, y_pred_t)



c:\Users\ashee\Desktop\crop project\venv\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 3.9135  | val_0_accuracy: 0.04091 |  0:00:00s
epoch 1  | loss: 3.40034 | val_0_accuracy: 0.08409 |  0:00:00s
epoch 2  | loss: 3.12377 | val_0_accuracy: 0.05682 |  0:00:00s
epoch 3  | loss: 2.96955 | val_0_accuracy: 0.05682 |  0:00:00s
epoch 4  | loss: 2.82263 | val_0_accuracy: 0.075   |  0:00:00s
epoch 5  | loss: 2.70233 | val_0_accuracy: 0.10227 |  0:00:00s
epoch 6  | loss: 2.64295 | val_0_accuracy: 0.09545 |  0:00:00s
epoch 7  | loss: 2.555   | val_0_accuracy: 0.07045 |  0:00:00s
epoch 8  | loss: 2.5162  | val_0_accuracy: 0.06136 |  0:00:00s
epoch 9  | loss: 2.43557 | val_0_accuracy: 0.07273 |  0:00:00s
epoch 10 | loss: 2.35808 | val_0_accuracy: 0.06364 |  0:00:00s
epoch 11 | loss: 2.28824 | val_0_accuracy: 0.06591 |  0:00:00s
epoch 12 | loss: 2.23772 | val_0_accuracy: 0.06591 |  0:00:00s
epoch 13 | loss: 2.17381 | val_0_accuracy: 0.07273 |  0:00:00s
epoch 14 | loss: 2.11166 | val_0_accuracy: 0.07045 |  0:00:00s
epoch 15 | loss: 2.07146 | val_0_accuracy: 0.05909 |  0

c:\Users\ashee\Desktop\crop project\venv\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [8]:
#Evaluate TabNet
y_pred_t = tabnet.predict(X_test_t)
from sklearn.metrics import accuracy_score, classification_report

acc_tabnet = accuracy_score(y_test_t, y_pred_t)
print("TabNet Accuracy:", acc_tabnet)
print(classification_report(y_test_t, y_pred_t, target_names=le.classes_))


TabNet Accuracy: 0.10227272727272728
              precision    recall  f1-score   support

       apple       0.09      0.60      0.16        20
      banana       0.00      0.00      0.00        20
   blackgram       0.00      0.00      0.00        20
    chickpea       0.00      0.00      0.00        20
     coconut       0.00      0.00      0.00        20
      coffee       0.00      0.00      0.00        20
      cotton       0.00      0.00      0.00        20
      grapes       0.00      0.00      0.00        20
        jute       0.00      0.00      0.00        20
 kidneybeans       0.25      0.05      0.08        20
      lentil       0.00      0.00      0.00        20
       maize       0.00      0.00      0.00        20
       mango       0.00      0.00      0.00        20
   mothbeans       0.00      0.00      0.00        20
    mungbean       0.26      0.80      0.40        20
   muskmelon       0.00      0.00      0.00        20
      orange       0.08      0.80      0.14 

c:\Users\ashee\Desktop\crop project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ashee\Desktop\crop project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ashee\Desktop\crop project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is

In [11]:
# reshape data for Transformer

import numpy as np

input_dim = X_train.shape[1]        
num_classes = y_train.shape[1]     

X_train_tr = X_train.reshape(-1, input_dim, 1)
X_test_tr  = X_test.reshape(-1, input_dim, 1)

print("Transformer input shape:", X_train_tr.shape)


Transformer input shape: (1760, 7, 1)


In [12]:
#Build Transformer model


from tensorflow.keras import layers, models

d_model = 32    
num_heads = 4   

inp = layers.Input(shape=(input_dim, 1)) 


x = layers.Dense(d_model)(inp) 


attn_out = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)(
    x, x
)
x = layers.Add()([x, attn_out])         
x = layers.LayerNormalization()(x)


ffn = layers.Dense(64, activation='relu')(x)
ffn = layers.Dense(d_model, activation='relu')(ffn)
x = layers.Add()([x, ffn])             
x = layers.LayerNormalization()(x)


x = layers.GlobalAveragePooling1D()(x)


x = layers.Dense(64, activation='relu')(x)
out = layers.Dense(num_classes, activation='softmax')(x)

transformer_model = models.Model(inputs=inp, outputs=out)

transformer_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

transformer_model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 7, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 7, 32)     │         64 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 7, 32)     │     16,800 │ dense_3[0][0],    │
│ (MultiHeadAttentio… │                   │            │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 7, 32)     │          0 │ dense_3[0][0],    │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 7, 32)     │         64 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 7, 64)     │      2,112 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 7, 32)     │      2,080 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 7, 32)     │          0 │ layer_normalizat… │
│                     │                   │            │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 7, 32)     │         64 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 64)        │      2,112 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 22)        │      1,430 │ dense_6[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,726 (96.59 KB)

 Trainable params: 24,726 (96.59 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
#Train Transformer model

from tensorflow.keras.callbacks import EarlyStopping

es_tr = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

history_tr = transformer_model.fit(
    X_train_tr, y_train,
    validation_split=0.1,
    epochs=80,
    batch_size=32,
    callbacks=[es_tr],
    verbose=2
)

print("Transformer-like model evaluation:")
acc_trans = evaluate(transformer_model, X_test_tr, y_test)


Epoch 1/80
50/50 - 3s - 50ms/step - accuracy: 0.1364 - loss: 2.8205 - val_accuracy: 0.1420 - val_loss: 2.5832
Epoch 2/80
50/50 - 0s - 6ms/step - accuracy: 0.2027 - loss: 2.3950 - val_accuracy: 0.2670 - val_loss: 2.2543
Epoch 3/80
50/50 - 0s - 5ms/step - accuracy: 0.3163 - loss: 2.1213 - val_accuracy: 0.2841 - val_loss: 2.0162
Epoch 4/80
50/50 - 0s - 5ms/step - accuracy: 0.3504 - loss: 1.8994 - val_accuracy: 0.3864 - val_loss: 1.8084
Epoch 5/80
50/50 - 0s - 6ms/step - accuracy: 0.4255 - loss: 1.6914 - val_accuracy: 0.4432 - val_loss: 1.6222
Epoch 6/80
50/50 - 0s - 5ms/step - accuracy: 0.4514 - loss: 1.5617 - val_accuracy: 0.4545 - val_loss: 1.5151
Epoch 7/80
50/50 - 0s - 5ms/step - accuracy: 0.4773 - loss: 1.4498 - val_accuracy: 0.5000 - val_loss: 1.4253
Epoch 8/80
50/50 - 0s - 5ms/step - accuracy: 0.5120 - loss: 1.3863 - val_accuracy: 0.4773 - val_loss: 1.3656
Epoch 9/80
50/50 - 0s - 5ms/step - accuracy: 0.5057 - loss: 1.3450 - val_accuracy: 0.5227 - val_loss: 1.3172
Epoch 10/80
50/50 

In [17]:
# ALGO 4 — Autoencoder + DNN

from tensorflow.keras import layers, models

encoding_dim = 4

input_layer = layers.Input(shape=(input_dim,))
encoded = layers.Dense(32, activation='relu')(input_layer)
encoded = layers.Dense(encoding_dim, activation='relu')(encoded)

decoded = layers.Dense(32, activation='relu')(encoded)
decoded = layers.Dense(input_dim, activation='linear')(decoded)

autoencoder = models.Model(input_layer, decoded)
encoder = models.Model(input_layer, encoded)

autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.fit(
    X_train, X_train,
    validation_split=0.1,
    epochs=80,
    batch_size=32,
    verbose=2
)



Epoch 1/80
50/50 - 1s - 17ms/step - loss: 0.9205 - val_loss: 0.9112
Epoch 2/80
50/50 - 0s - 3ms/step - loss: 0.7800 - val_loss: 0.7636
Epoch 3/80
50/50 - 0s - 3ms/step - loss: 0.6337 - val_loss: 0.5663
Epoch 4/80
50/50 - 0s - 3ms/step - loss: 0.4874 - val_loss: 0.4526
Epoch 5/80
50/50 - 0s - 3ms/step - loss: 0.4069 - val_loss: 0.3738
Epoch 6/80
50/50 - 0s - 3ms/step - loss: 0.3295 - val_loss: 0.2991
Epoch 7/80
50/50 - 0s - 4ms/step - loss: 0.2727 - val_loss: 0.2536
Epoch 8/80
50/50 - 0s - 3ms/step - loss: 0.2320 - val_loss: 0.2221
Epoch 9/80
50/50 - 0s - 3ms/step - loss: 0.2042 - val_loss: 0.1958
Epoch 10/80
50/50 - 0s - 3ms/step - loss: 0.1829 - val_loss: 0.1787
Epoch 11/80
50/50 - 0s - 3ms/step - loss: 0.1679 - val_loss: 0.1686
Epoch 12/80
50/50 - 0s - 3ms/step - loss: 0.1572 - val_loss: 0.1568
Epoch 13/80
50/50 - 0s - 3ms/step - loss: 0.1496 - val_loss: 0.1511
Epoch 14/80
50/50 - 0s - 3ms/step - loss: 0.1429 - val_loss: 0.1445
Epoch 15/80
50/50 - 0s - 3ms/step - loss: 0.1363 - val_l

In [18]:
#Classifier Using Encoded Features
X_train_enc = encoder.predict(X_train)
X_test_enc = encoder.predict(X_test)

clf = models.Sequential([
    layers.Dense(32, activation='relu', input_shape=(encoding_dim,)),
    layers.Dense(num_classes, activation='softmax')
])

clf.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

clf.fit(
    X_train_enc, y_train,
    validation_split=0.1,
    epochs=80,
    batch_size=32,
    verbose=2
)

acc_auto = evaluate(clf, X_test_enc, y_test)


55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 855us/step
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
Epoch 1/80


c:\Users\ashee\Desktop\crop project\venv\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


50/50 - 1s - 14ms/step - accuracy: 0.0878 - loss: 3.0264 - val_accuracy: 0.2045 - val_loss: 2.7623
Epoch 2/80
50/50 - 0s - 3ms/step - accuracy: 0.2626 - loss: 2.6291 - val_accuracy: 0.3239 - val_loss: 2.4127
Epoch 3/80
50/50 - 0s - 4ms/step - accuracy: 0.3163 - loss: 2.3437 - val_accuracy: 0.3920 - val_loss: 2.1325
Epoch 4/80
50/50 - 0s - 4ms/step - accuracy: 0.3769 - loss: 2.0987 - val_accuracy: 0.4886 - val_loss: 1.9098
Epoch 5/80
50/50 - 0s - 3ms/step - accuracy: 0.4583 - loss: 1.8993 - val_accuracy: 0.5114 - val_loss: 1.7313
Epoch 6/80
50/50 - 0s - 3ms/step - accuracy: 0.5158 - loss: 1.7213 - val_accuracy: 0.5852 - val_loss: 1.5776
Epoch 7/80
50/50 - 0s - 3ms/step - accuracy: 0.5966 - loss: 1.5739 - val_accuracy: 0.6136 - val_loss: 1.4353
Epoch 8/80
50/50 - 0s - 3ms/step - accuracy: 0.6042 - loss: 1.4465 - val_accuracy: 0.6250 - val_loss: 1.3275
Epoch 9/80
50/50 - 0s - 3ms/step - accuracy: 0.6755 - loss: 1.3355 - val_accuracy: 0.6534 - val_loss: 1.2248
Epoch 10/80
50/50 - 0s - 3ms/

In [20]:
#COMPARE ALL ALGORITHMS
# Compare all algorithms
results = {
    "DNN / MLP": acc_mlp,
    "TabNet": acc_tabnet,
    "Autoencoder + DNN": acc_auto,
    "Keras Transformer": acc_trans
}

import pandas as pd
pd.DataFrame(results, index=["Accuracy"]).T.sort_values(by="Accuracy", ascending=False)



,Accuracy
DNN / MLP,0.984091
Autoencoder + DNN,0.918182
Keras Transformer,0.688636
TabNet,0.102273


In [21]:
#TEST NEW VALUE ON DNN / MLP
sample = {
    'N': 50,
    'P': 40,
    'K': 40,
    'temperature': 25,
    'humidity': 70,
    'ph': 6.5,
    'rainfall': 150
}

import pandas as pd
import numpy as np

df_sample = pd.DataFrame([sample])
scaled = scaler.transform(df_sample)

pred_prob = mlp.predict(scaled)
pred_class = np.argmax(pred_prob)
print("Predicted Crop:", le.inverse_transform([pred_class])[0])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
Predicted Crop: jute


In [22]:
#TEST NEW VALUE ON TABNet
sample = [
    sample['N'], sample['P'], sample['K'], 
    sample['temperature'], sample['humidity'], 
    sample['ph'], sample['rainfall']
]

import numpy as np
tab_pred = tabnet.predict(np.array([sample]))
print("Predicted Crop (TabNet):", le.inverse_transform(tab_pred)[0])


Predicted Crop (TabNet): orange


In [23]:
#TEST NEW VALUE ON Autoencoder+DNN
df_sample = pd.DataFrame([sample])
scaled = scaler.transform(df_sample)

# Encode
encoded = encoder.predict(scaled)

# Classify
pred_prob = clf.predict(encoded)
pred_class = np.argmax(pred_prob)
print("Predicted Crop (Autoencoder+DNN):", le.inverse_transform([pred_class])[0])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
Predicted Crop (Autoencoder+DNN): jute


c:\Users\ashee\Desktop\crop project\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [24]:
#TEST NEW VALUE ON Keras Transformer
df_sample = pd.DataFrame([sample])
scaled = scaler.transform(df_sample)

# reshape for transformer (batch, seq_len, channels)
trans_input = scaled.reshape(1, len(sample), 1)

pred_prob = transformer_model.predict(trans_input)
pred_class = np.argmax(pred_prob)
print("Predicted Crop (Transformer):", le.inverse_transform([pred_class])[0])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
Predicted Crop (Transformer): maize


c:\Users\ashee\Desktop\crop project\venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [25]:
#Testing Function
def test_value(model, sample_dict, model_type):
    df_sample = pd.DataFrame([sample_dict])

    if model_type == "tabnet":
        arr = df_sample.values
        pred = model.predict(arr)
        return le.inverse_transform(pred)[0]

    if model_type == "transformer":
        scaled = scaler.transform(df_sample)
        reshaped = scaled.reshape(1, len(sample_dict), 1)
        pred = model.predict(reshaped)
        return le.inverse_transform([np.argmax(pred)])[0]

    if model_type == "autoencoder":
        scaled = scaler.transform(df_sample)
        encoded = encoder.predict(scaled)
        pred = clf.predict(encoded)
        return le.inverse_transform([np.argmax(pred)])[0]

    # default for MLP/DNN
    scaled = scaler.transform(df_sample)
    pred = model.predict(scaled)
    return le.inverse_transform([np.argmax(pred)])[0]
